In [1]:
import os
import pandas as pd
import json

I started my data tranformation by using pandas to create a data table out of the information I oulled during my data collection. I then print the shape and first few values to check for accuracy.

In [2]:
with open("data/raw/openlibrary_fiction_2016_2025.json", "r") as f:
    docs = json.load(f)

df = pd.DataFrame(docs)

print("Shape:", df.shape)
print(df.dtypes)
df.head()

Shape: (5000, 6)
author_name                object
edition_key                object
first_publish_year          int64
key                           str
number_of_pages_median    float64
title                         str
dtype: object


,author_name,edition_key,first_publish_year,key,number_of_pages_median,title
0,[Amor Towles],"[OL35368976M, OL32630673M, OL39624052M, OL3535...",2016,/works/OL17797130W,511.0,A Gentleman in Moscow
1,[Penelope Douglas],"[OL49950513M, OL43363145M, OL43363112M, OL4202...",2016,/works/OL20808172W,363.0,Punk 57
2,[Neal Shusterman],"[OL27931281M, OL47067218M, OL28209389M, OL2818...",2016,/works/OL17876096W,449.0,Scythe
3,[Sarah J. Maas],"[OL43631860M, OL51687001M, OL52211184M, OL4923...",2016,/works/OL17625829W,704.0,Empire of Storms
4,[Rick Riordan],"[OL60429529M, OL28109957M, OL28274365M, OL4366...",2016,/works/OL17364743W,384.0,The Hidden Oracle


My next step of my data tranformation required outside research. I used isnull to create a dataframe of the exact same shape as the original but with true/false values. It was true if there was no value, false if there was a value. I did it this way becayse true counts as 1 and false counts as zero. I then summed all the values to get the total number of missing values and printed the result.

In [3]:
print(df.isnull().sum())
print(f"\nMissing page count: {df['number_of_pages_median'].isnull().sum()} / {len(df)} "
      f"({df['number_of_pages_median'].isnull().mean()*100:.1f}%)")

author_name                 5
edition_key                 0
first_publish_year          0
key                         0
number_of_pages_median    957
title                       0
dtype: int64

Missing page count: 957 / 5000 (19.1%)


My next step was to print out the summary statistics of my data.

In [3]:
print(df['number_of_pages_median'].describe())

count    4043.000000
mean      361.851843
std       161.104518
min         1.000000
25%       288.000000
50%       352.000000
75%       416.000000
max      3504.000000
Name: number_of_pages_median, dtype: float64


I then printed the 10 shortrst and longest books from my collected data. This was very helpful because it helped me realize that some books with page counts on the higher and lower end are actually misclassifications of pamplets and book series that were being counted as a single book. I played with the quantity that I printed to see the full extent of how many books were being misclassified.

In [12]:

print("Shortest:")
print(df.nsmallest(10, 'number_of_pages_median')[['title', 'first_publish_year', 'number_of_pages_median']])

print("\nLongest:")
print(df.nlargest(50, 'number_of_pages_median')[['title', 'first_publish_year', 'number_of_pages_median']])

Shortest:
                       title  first_publish_year  number_of_pages_median
392        Highland Spitfire                2016                     1.0
715             The Cartel 7                2017                     1.0
950       Powers of darkness                2017                     1.0
1382     Next year in Havana                2018                     1.0
1702              Fix Her Up                2019                     1.0
1729  A Good Kind of Trouble                2019                     1.0
2199    Love Her or Lose Her                2020                     1.0
2205                Lakewood                2020                     1.0
2270               Riot baby                2020                     1.0
2930    The Box in the Woods                2021                     1.0

Longest:
                                                  title  first_publish_year  \
4037  King of Wrath / King of Pride / King of Greed ...                2024   
3927               

My next step was to set hard limits for page minimum and maximum. I selected 50 as the minimum and 900 as the maximum page count because that was about where the misclassifications seemed to stop when looking at my previous code block.

In [5]:
before = len(df)

df_clean = df[
    (df['number_of_pages_median'] >= 50) &
    (df['number_of_pages_median'] <= 900)  
].copy()

print(f"Dropped {before - len(df_clean)} rows outside 50–900 pages")
print(f"Remaining: {len(df_clean)} rows")

Dropped 1104 rows outside 50–900 pages
Remaining: 3896 rows


I then checked for duplicates and found none.

In [6]:
dupes = df_clean['key'].duplicated().sum()
print(f"Duplicate 'key' values: {dupes}")

Duplicate 'key' values: 0


I then made a copy of my original data frame that kept only the relevant columns. 

In [7]:
df_final = df_clean[['key', 'title', 'author_name', 'first_publish_year', 'number_of_pages_median']].copy()
df_final = df_final.rename(columns={'number_of_pages_median': 'page_count'})

print(df_final.shape)
df_final.head()

(3896, 5)


,key,title,author_name,first_publish_year,page_count
0,/works/OL17797130W,A Gentleman in Moscow,[Amor Towles],2016,511.0
1,/works/OL20808172W,Punk 57,[Penelope Douglas],2016,363.0
2,/works/OL17876096W,Scythe,[Neal Shusterman],2016,449.0
3,/works/OL17625829W,Empire of Storms,[Sarah J. Maas],2016,704.0
4,/works/OL17364743W,The Hidden Oracle,[Rick Riordan],2016,384.0


I then created the file path, saved my data as a csv file, and removed the indexing to simplify my data frame.

In [8]:
os.makedirs("data/processed", exist_ok=True)

df_final.to_csv("data/processed/fiction_pagecounts_2016_2025.csv", index=False)
print("Saved to data/processed/fiction_pagecounts_2016_2025.csv")

Saved to data/processed/fiction_pagecounts_2016_2025.csv
